In [1]:
import os
import shutil
import random
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

In [2]:
# 1. SET PATHS
# Adjust if your segmented folder is named differently
SEGMENTED_PATH = r"D:\Development\8th Sem Project\TomatoClassification\dataset\segmented"
OUTPUT_PATH    = r"D:\Development\8th Sem Project\TomatoClassification\dataset\processed_segmented"

IMAGE_SIZE = (224, 224)
MAX_IMAGES_PER_CLASS = 1000
SEED = 42

In [3]:
# 2. GET CLASSES (Ensuring match with Step 3)
TOMATO_CLASSES = sorted([
    f for f in os.listdir(SEGMENTED_PATH) 
    if f.startswith("Tomato") and os.path.isdir(os.path.join(SEGMENTED_PATH, f))
])

print(f"✅ Found {len(TOMATO_CLASSES)} classes in segmented folder.")

✅ Found 10 classes in segmented folder.


In [4]:
# Run this BEFORE the processing cell
print("📊 Segmented Dataset Summary\n" + "="*40)
class_counts = {}
for cls in TOMATO_CLASSES:
    cls_path = os.path.join(SEGMENTED_PATH, cls)
    images = [f for f in os.listdir(cls_path)
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    class_counts[cls] = len(images)
    print(f"✅ {cls.replace('Tomato___', ''):<45} → {len(images)} images")

print("="*40)
print(f"📁 Total Classes : {len(class_counts)}")
print(f"🖼️  Total Images  : {sum(class_counts.values())}")

📊 Segmented Dataset Summary
✅ Bacterial_spot                                → 2127 images
✅ Early_blight                                  → 1000 images
✅ Late_blight                                   → 1909 images
✅ Leaf_Mold                                     → 952 images
✅ Septoria_leaf_spot                            → 1771 images
✅ Spider_mites Two-spotted_spider_mite          → 1676 images
✅ Target_Spot                                   → 1404 images
✅ Tomato_Yellow_Leaf_Curl_Virus                 → 5357 images
✅ Tomato_mosaic_virus                           → 373 images
✅ healthy                                       → 1591 images
📁 Total Classes : 10
🖼️  Total Images  : 18160


In [5]:
# 3. PROCESSING & SPLITTING
split_counts = {'train': {}, 'val': {}, 'test': {}}

for cls in TOMATO_CLASSES:
    cls_path = os.path.join(SEGMENTED_PATH, cls)
    all_images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    # Cap at 1000 and use Seed 42 for consistency
    if len(all_images) > MAX_IMAGES_PER_CLASS:
        random.seed(SEED)
        all_images = random.sample(all_images, MAX_IMAGES_PER_CLASS)
    
    # Split 70/15/15
    train_imgs, temp_imgs = train_test_split(all_images, test_size=0.30, random_state=SEED)
    val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.50, random_state=SEED)
    
    splits = {'train': train_imgs, 'val': val_imgs, 'test': test_imgs}
    
    for split_name, img_list in splits.items():
        dest_dir = os.path.join(OUTPUT_PATH, split_name, cls)
        os.makedirs(dest_dir, exist_ok=True)
        split_counts[split_name][cls] = len(img_list)
        
        for img_name in img_list:
            src = os.path.join(cls_path, img_name)
            dst = os.path.join(dest_dir, img_name)
            
            # Use same resize logic as Phase 1
            img = Image.open(src).convert("RGB")
            img = img.resize(IMAGE_SIZE, Image.LANCZOS)
            img.save(dst)

print("\n✅ Segmented Preprocessing Complete!")
print(f"📂 Saved to: {OUTPUT_PATH}")


✅ Segmented Preprocessing Complete!
📂 Saved to: D:\Development\8th Sem Project\TomatoClassification\dataset\processed_segmented


In [6]:
# Add this after your existing print statements
print(f"\n{'Class':<45} {'Train':>8} {'Val':>8} {'Test':>8}")
print("-" * 72)
for cls in TOMATO_CLASSES:
    name = cls.replace("Tomato___", "")
    t  = split_counts['train'].get(cls, 0)
    v  = split_counts['val'].get(cls, 0)
    te = split_counts['test'].get(cls, 0)
    print(f"{name:<45} {t:>8} {v:>8} {te:>8}")

print("-" * 72)
print(f"{'TOTAL':<45} "
      f"{sum(split_counts['train'].values()):>8} "
      f"{sum(split_counts['val'].values()):>8} "
      f"{sum(split_counts['test'].values()):>8}")


Class                                            Train      Val     Test
------------------------------------------------------------------------
Bacterial_spot                                     700      150      150
Early_blight                                       700      150      150
Late_blight                                        700      150      150
Leaf_Mold                                          666      143      143
Septoria_leaf_spot                                 700      150      150
Spider_mites Two-spotted_spider_mite               700      150      150
Target_Spot                                        700      150      150
Tomato_Yellow_Leaf_Curl_Virus                      700      150      150
Tomato_mosaic_virus                                261       56       56
healthy                                            700      150      150
------------------------------------------------------------------------
TOTAL                                             